Word2Vec(sentences, vector_size=10, window=1, min_count=1, sg=0)
- sentences: 학습할 데이터
- vector_size=10: 하나의 단어를 몇 개의 차원으로 표현할 것인가, 단어 하나당 10개의 숫자로 이루어진 벡터 생성
- window=1: 중심 단어를 맞추기 위해 주변 단어를 앞뒤로 몇 개까지 볼 것인가
- min_count=1: 단어의 최소 등장 횟수, 1로 설정했으므로, 단 한 번만 등장한 단어('오늘', '좋다' 등)도 무시하지 않고 전부 학습 단어장에 포함
- sg=0: 학습 알고리즘을 선택, sg는 Skip-Gram의 약자
    - 0: CBOW
    - 1: Skip-Gram

In [24]:
#CBOW는 주변 단어(Context)를 통해 중간 단어(Target)을 예측, sg=0 옵션을 사용
from gensim.models import Word2Vec

# 주변 단어로 중심 단어를 예측하는 데이터셋
sentences = [["오늘", "날씨", "정말", "좋다"], ["내일", "날씨", "어떨까", "궁금해"]]

# sg=0 이 CBOW 모델입니다.
cbow_model = Word2Vec(sentences, vector_size=5, window=1, min_count=1, sg=0)

print("CBOW '날씨' 벡터:", cbow_model.wv['날씨'])

CBOW '날씨' 벡터: [-0.01072454  0.00472863  0.10206699  0.18018547 -0.186059  ]


## Word2Vec -> NumPy

In [ ]:
import numpy as np

sentences = [["오늘", "날씨", "정말", "좋다"], ["내일", "날씨", "어떨까", "궁금해"]]
print("sentences: ", sentences)
#1. 단어 사전 만들기
vocab = list(set([word for sentence in sentences for word in sentence]))
vocab_size = len(vocab)

#2. 각 단어에 번호 붙이기
word_to_idx = {word: i for i, word in enumerate(vocab)}
# {'정말': 0, '날씨': 1, '궁금해': 2, '어떨까': 3, '내일': 4, '오늘': 5, '좋다': 6}
idx_to_word = {i: word for i, word in enumerate(vocab)}
# {0: '정말', 1: '날씨', 2: '궁금해', 3: '어떨까', 4: '내일', 5: '오늘', 6: '좋다'}

#3. 하이퍼파라미터 설정
vector_size = 5         # 단어를 5개 차원? 숫자로 표현
learning_rate = 0.1     # 모델이 한 번 학습하는 가중치 학습률
epochs = 1000            # 전체 학습 데이터를 총 몇 번 반복해서 학습할지 에포크 결정

#4. 가중치 초기화
# W1: 입력층에서 은닉층으로 가는 가중치 행렬, 무작위 값으로 초기화
# 학습이 완료되면 이 W1 행렬의 각 행이 바로 각 단어의 임베딩 벡터
W1 = np.random.randn(vocab_size, vector_size)
# W2: 은닉층에서 출력층으로 가는 가중치 행열, 무작위 값으로 초기화
W2 = np.random.randn(vector_size, vocab_size)

#5. Utility
# sofrmax: 점수를 확률처럼 변경해주는 함수
def softmax(x):
    e = np.exp(x-np.max(x)) # exp 계산 시 값이 너무 커져 컴퓨터가 계산하지 못하는 overflow를 막기 위해 최댓값 빼기
    return e / np.sum(e)

#6. 학습 데이터 생성
training_data = []

for sentence in sentences:
    indices = [word_to_idx[w] for w in sentence]    # indices:  [5, 1, 0, 6] / indices:  [4, 1, 3, 2]
    print("indices: ", indices)

    for i in range (1, len(sentence) -1):
        context = [word_to_idx[sentence[i-1]], word_to_idx[sentence[i+1]]]
        center = word_to_idx[sentence[i]]
        training_data.append((context, center))     # training_data:  [([5, 0], 1), ([1, 6], 0), ([4, 3], 1), ([1, 2], 3)]
print("training_data: ", training_data)

for context, center in training_data:
    context_words = [idx_to_word[idx] for idx in context]
    center_words = idx_to_word[center]
    print(context_words, '->', center_words)
print('*'*100)

#7. 모델 학습 루프
for epoch in range(epochs):
    loss = 0
    for context, center in training_data:
        # 순전파
        # 은닉층의 값을 담을 크기가 0 벡터 만들기
        h = np.zeros(vector_size)

        for c in context:
            h += W1[c]          # W1 행렬에서 해당 주변 단어의 인덱스 위치에 있는 행을 그대로 꺼내어 h에 합산

        h = h / len(context)    # CBOB의 핵심: 주변 단어 벡터들을 모두 더한 값을 주변 단어의 개수로 나누어 평균 벡터 만들기
        u = np.dot(W2.T, h)     # 은닉층의 평균 벡터(h)와 가중치 W2를 내적하여 각 단어별 출력 스코어(u)를 계산
        y_pred = softmax(u)     # 스코어(u)를 소프트맥스 함수에 통과시켜 '어떤 단어가 중심 단어일 확률이 높은지' 확률 분포(y_pred)를 생성

        #8. 손실함수 계산
        # 예측한 확률 분포(y_pred) 중 실제 정답 단어(center)의 확률값에 로그를 취하여 오차에 누적
        # 1e-10는 확률이 0이 되어 로그 값이 무한대로 가는 것을 방지하기 위한 아주 작은 숫자
        loss -= np.log(y_pred[center] + 1e-10)
        
        #9. 역전파
        err = y_pred.copy()     # 출력층의 오차를 구하기 위해 예측 확률 분포(y_pred)를 그대로 복사
        err[center] -= 1        # 실제 정답 단어의 위치에 해당하는 확률값에서 1을 빼기

        dW2 = np.outer(h, err)  # 은닉층 벡터(h)와 출력층 오차(err)를 외적하여 W2 가중치가 변해야 할 방향과 크기(기울기 dW2)를 계산
        dh = np.dot(W2, err)    # W2 가중치와 출력층 오차(err)를 내적하여 은닉층으로 거꾸로 전달될 오차(dh)를 계산

        dW1 = np.zeros_like(W1) # W1 가중치가 변해야 할 기울기(dW1)를 담을 0으로 채워진 행렬을 W1과 같은 크기로 만들기
        
    
        for c in context:       # 순전파 때 사용한 주변 단어(context)들
            dW1[c] += dh / len(context) # 순전파 때 벡터를 합치고 개수만큼 '평균'을 냈으므로, 오차(dh)도 똑같이 개수만큼 나누어 W1의 각 단어 기울기에 배분
            
            
        # 10. 가중치 업데이트 Gradient Descent
        W1 -= learning_rate * dW1       # 계산된 W1의 기울기(dW1)에 학습률을 곱한 값을 기존 W1에서 빼주어 가중치 업데이트.
        W2 -= learning_rate * dW2       # 계산된 W2의 기울기(dW2)에 학습률을 곱한 값을 기존 W2에서 빼주어 가중치 업데이트

    if (epoch + 1) % 200 == 0:          # 200번의 에포크가 끝날 때마다 현재까지 누적된 오차(Loss) 출력
        print(f"Epoch {epoch + 1}, Loss: {loss:.4f}")

# 11. 최종 단어 벡터 추출
print("\nNumPy로 학습된 CBOW 벡터:")
for word in vocab:
      print(word, W1[word_to_idx[word]])

sentences:  [['오늘', '날씨', '정말', '좋다'], ['내일', '날씨', '어떨까', '궁금해']]
indices:  [5, 1, 0, 6]
indices:  [4, 1, 3, 2]
training_data:  [([5, 0], 1), ([1, 6], 0), ([4, 3], 1), ([1, 2], 3)]
['오늘', '정말'] -> 날씨
['날씨', '좋다'] -> 정말
['내일', '어떨까'] -> 날씨
['날씨', '궁금해'] -> 어떨까
****************************************************************************************************
Epoch 200, Loss: 0.0192
Epoch 400, Loss: 0.0081
Epoch 600, Loss: 0.0050
Epoch 800, Loss: 0.0036
Epoch 1000, Loss: 0.0028

NumPy로 학습된 CBOW 벡터:
정말 [ 0.10134268  2.06488534  0.42159026  2.44286253 -0.10693484]
날씨 [ 0.20561745  0.65971758 -3.05293536 -0.57565548 -0.14429642]
궁금해 [-1.75394807  2.23336376 -1.43392046 -2.78898034 -1.04358427]
어떨까 [-0.4866895   2.07414952  0.6592732  -0.18309462  0.49858258]
내일 [-1.09098801  1.38726752  0.39399718  2.49500062  0.07374814]
오늘 [ 0.76478674  2.9345664  -0.30634696  0.27403657  0.49572953]
좋다 [ 1.71114086 -1.2473305  -0.65270679  1.15560656 -2.0039839 ]
